# Kashmiri Whisper v7 — balanced replay candidate
Conservatively continues v4 for one low-learning-rate epoch on capped Meta, Bolbosh, and OpenSLR replay data. Production is not modified by this notebook.

In [ ]:
!nvidia-smi
!pip uninstall -y -q torchao
!pip install -q 'transformers==4.48.3' 'peft==0.14.0' 'accelerate==1.3.0' 'datasets==3.2.0' 'evaluate>=0.4' soundfile librosa jiwer huggingface_hub pyarrow
!git clone -q --depth 1 https://github.com/sameerahmad86232/cipher.git /kaggle/working/cipher

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    print('HF_TOKEN unavailable: using public OpenSLR only; gated AI4Bharat will be skipped.')
include_gated = '--include-gated' if os.getenv('HF_TOKEN') else ''
!python /kaggle/working/cipher/scripts/prepare-public-kashmiri-asr.py --output /kaggle/working/v7-balanced-speech --max-per-source 1000 --source-id facebook/omnilingual-asr-corpus --source-id GAASH-Lab/Bolbosh --source-id programindz/kashmiri-audio-corpus
import pandas as pd
manifest = pd.read_csv('/kaggle/working/v7-balanced-speech/metadata.csv')
print(manifest.groupby(['source','license']).agg(clips=('filename','size'), hours=('duration',lambda x: x.sum()/3600)))
assert len(manifest) >= 1800, 'Insufficient balanced real speech; refusing to train'

In [ ]:
# Download model artifacts explicitly so a stalled network transfer is visible and retryable.
import os
from huggingface_hub import snapshot_download
os.environ['HF_HUB_ETAG_TIMEOUT'] = '30'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
base_dir = snapshot_download('openai/whisper-large-v3-turbo', local_dir='/kaggle/working/whisper-large-v3-turbo', max_workers=4)
adapter_dir = snapshot_download('Sameer0313/kashmiri-whisper-lora-v4-candidate', local_dir='/kaggle/working/whisper-v4-adapter', max_workers=4)
print('Model artifacts ready:', base_dir, adapter_dir, flush=True)

In [ ]:
import os, subprocess, sys
env = os.environ.copy()
env.update({
  'BASE_MODEL':'/kaggle/working/whisper-large-v3-turbo',
  'STAGE1_ADAPTER':'/kaggle/working/whisper-v4-adapter',
  'NUM_EPOCHS':'1','BATCH_SIZE':'2','GRAD_ACCUM':'8','LEARNING_RATE':'2e-6',
  'EVAL_SIZE':'64','SAVE_STEPS':'20','ACOUSTIC_AUG_PROB':'0.35','SPEC_AUG_PROB':'0.35',
  'AUDIO_ROOTS_ONLY':'1','REQUIRE_REAL_AUDIO':'1',
  'PYTHONUNBUFFERED':'1','TOKENIZERS_PARALLELISM':'false',
  'EXTRA_AUDIO_ROOTS':'/kaggle/working/v7-balanced-speech',
  'OUTPUT_DIR':'/kaggle/working/kashmiri-whisper-v7-candidate',
  'CHECKPOINTS_DIR':'/kaggle/working/whisper-v7-checkpoints',
  'HF_REPO_ID':'Sameer0313/kashmiri-whisper-lora-v7-candidate'
})
subprocess.run([sys.executable,'-u','scripts/finetune-whisper-ultra-accuracy.py'],cwd='/kaggle/working/cipher',env=env,check=True,timeout=14400)

In [ ]:
from pathlib import Path
import json, shutil
manifest = Path('/kaggle/working/v7-balanced-speech/metadata.csv')
report = {'candidate':'Sameer0313/kashmiri-whisper-lora-v7-candidate','training_manifest_rows':sum(1 for _ in manifest.open())-1,'production_promoted':False,'requires_controlled_v4_v7_evaluation':True}
Path('/kaggle/working/whisper-v7-training-report.json').write_text(json.dumps(report,indent=2))
shutil.make_archive('/kaggle/working/kashmiri-whisper-v7-candidate','zip','/kaggle/working/kashmiri-whisper-v7-candidate')
print(report)